# Decode-parameter sweep (Kaggle)
Needs inputs: competition data, `filseg-code`, `filseg-ckpt` (with `checkpoints/baseline_f0/best.pth`). GPU T4, Internet On.

In [ ]:
from pathlib import Path

def find_input(name):
    root = Path("/kaggle/input")
    for pattern in (name, f"*/{name}", f"*/*/{name}", f"*/*/*/{name}"):
        for p in root.glob(pattern):
            if p.is_dir():
                return p
    return None

FOLD = 0

In [ ]:
import os, shutil, subprocess, sys

CODE = Path("/kaggle/tmp/code")
if CODE.exists():
    shutil.rmtree(CODE)
src = find_input("filseg-code")
assert src is not None, sorted(os.listdir("/kaggle/input"))
shutil.copytree(src, CODE)
os.chdir(CODE)
%pip install -q segmentation-models-pytorch albumentations pycocotools
os.environ["FILSEG_WORK"] = "/kaggle/tmp/filseg_work"
subprocess.run([sys.executable, "scripts/build_labels.py"], check=True)
subprocess.run([sys.executable, "scripts/make_folds.py"], check=True)
subprocess.run([sys.executable, "scripts/build_prep.py"], check=True)

In [ ]:
import re

ck_root = find_input("filseg-ckpt")
assert ck_root is not None, sorted(os.listdir("/kaggle/input"))
specs = []
for p in sorted(ck_root.rglob("best.pth")):
    m = re.search(r"_f(\d+)$", p.parent.name)
    if m:
        specs.append(f"{p}:{m.group(1)}")
assert specs, f"no best.pth with _fN parent under {ck_root}"
print("OOF specs:", specs)
subprocess.run([sys.executable, "scripts/sweep_decode.py", "--oof", *specs,
                "--tta", "--predict-test",
                "--out", "/kaggle/working/sweep_results.json"], check=True)
print(sorted(os.listdir("/kaggle/working")))